# Step 08 – Spatial autocorrelation of test errors (Moran's I, join counts)

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.6, 3.3 · Table 8, Figure 8
- **Needs:** step 04
- **Writes:** `RESULTS_DIR/morans_i_errors/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / morans_i_errors/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q esda libpysal

In [ ]:
"""
Spatial autocorrelation of classification errors on the test partition

Run AFTER step 04 (04_model_tuning_and_evaluation): it reads test_predictions.csv saved there.

For every model, each of the 840 test points gets an error indicator
(1 = misclassified, 0 = correct). Spatial clustering of that indicator is tested with:
  - global Moran's I (row-standardised distance-band weights, 999 permutations)
  - join-count statistic BB (binary weights): number of neighbouring pairs in which
    BOTH points are misclassified, compared with random relabelling
Several distance bands are tested; 10 km matches the spatial block size used in step 05.

Outputs (OUT_DIR):
  morans_i_errors_all.csv        every model x distance band
  morans_i_errors_table.csv      compact table for the manuscript (one band)
  Fig_morans_i_correlogram.(png|pdf)
  Fig_error_map_<model>.(png|pdf)

In Colab first:  !pip install -q esda libpysal
"""


import os, json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import DistanceBand, w_subset, min_threshold_distance
from esda.moran import Moran
from esda.join_counts import Join_Counts

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'   # outputs of step 04 (04_model_tuning_and_evaluation)
OUT_DIR    = RESULTS_DIR + 'morans_i_errors/'
LABEL_COL  = 'class'
SEED       = 42
PERMUTATIONS = 999
BANDS_KM   = [1, 2, 5, 10]          # 10 km = spatial block size used in step 05
TABLE_BAND_KM = 10                  # band reported in the manuscript table
MAIN_MODELS = ['CatBoost', 'GTB', 'XGBoost', 'MLP']
MAP_MODEL  = 'CatBoost'
CLASS_NAMES = {'0': 'Water', '1': 'Forest', '2': 'Bare ground', '3': 'Urban', '4': 'Crops/grass'}

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ coordinates
df = pd.read_csv(DATA_PATH)

def get_lonlat(frame):
    """GEE CSV exports store geometry as GeoJSON in '.geo'; fall back to lon/lat columns."""
    if '.geo' in frame.columns:
        return np.array([json.loads(g)['coordinates'] for g in frame['.geo']], dtype=float)
    for lon, lat in [('longitude', 'latitude'), ('lon', 'lat'), ('Longitude', 'Latitude'), ('x', 'y')]:
        if lon in frame.columns and lat in frame.columns:
            return frame[[lon, lat]].to_numpy(dtype=float)
    raise ValueError(f'No coordinates found. Columns: {frame.columns.tolist()}')

lonlat = get_lonlat(df)
gdf = gpd.GeoDataFrame(df[[LABEL_COL]].copy(),
                       geometry=gpd.points_from_xy(lonlat[:, 0], lonlat[:, 1]), crs='EPSG:4326')
gdf = gdf.to_crs('EPSG:32629')                # same UTM 29N grid as the GEE export and step 05
print('Projected CRS:', gdf.crs)

# ------------------------------------------------------------------ test points and errors
pred = pd.read_csv(MODEL_DIR + 'test_predictions.csv')
test = gdf.loc[pred['row_idx'].values]
assert (test[LABEL_COL].astype(str).values == pred['y_true'].astype(str).values).all(), \
    'Labels in test_predictions.csv do not match the data file - check row indices'

coords = np.column_stack([test.geometry.x.values, test.geometry.y.values])
n = len(coords)
n_dup = pd.DataFrame(coords).duplicated().sum()
d_min = min_threshold_distance(coords)
print(f'Test points: {n} | duplicate coordinates: {n_dup} | '
      f'distance at which every point has >= 1 neighbour: {d_min/1000:.2f} km')

models = [c for c in pred.columns if c not in ('row_idx', 'y_true')]
y_true = pred['y_true'].astype(str).values
errors = {m: (pred[m].astype(str).values != y_true).astype(int) for m in models}

# ------------------------------------------------------------------ Moran's I and join counts
rows = []
for band_km in BANDS_KM:
    w = DistanceBand(coords, threshold=band_km * 1000, binary=True, silence_warnings=True)
    keep = [i for i in w.id_order if w.cardinalities[i] > 0]     # drop islands (no neighbour)
    if len(keep) < n:
        w = w_subset(w, keep, silence_warnings=True)
    mean_nb = np.mean([w.cardinalities[i] for i in w.id_order])
    print(f'\nBand {band_km} km: {len(keep)} points with neighbours '
          f'({n - len(keep)} islands dropped) | mean neighbours {mean_nb:.1f}')

    for m in models:
        e = errors[m][keep]
        row = {'Model': m, 'band_km': band_km, 'n_points': len(keep),
               'n_islands': n - len(keep), 'mean_neighbours': mean_nb,
               'n_errors': int(e.sum()), 'error_rate': e.mean()}
        if 2 <= e.sum() <= len(e) - 2:
            w.transform = 'B'
            np.random.seed(SEED)
            jc = Join_Counts(e, w, permutations=PERMUTATIONS)
            w.transform = 'R'
            np.random.seed(SEED)
            mi = Moran(e, w, transformation='r', permutations=PERMUTATIONS)
            row.update({'I': mi.I, 'E_I': mi.EI, 'z_sim': mi.z_sim, 'p_sim': mi.p_sim,
                        'BB_obs': jc.bb, 'BB_exp': jc.mean_bb, 'p_sim_BB': jc.p_sim_bb})
        rows.append(row)
        if m in MAIN_MODELS and 'I' in row:
            print(f"   {m:9s} errors {row['n_errors']:3d} | I = {row['I']:+.4f} "
                  f"(p = {row['p_sim']:.3f}) | BB {row['BB_obs']:.0f} vs {row['BB_exp']:.1f} "
                  f"expected (p = {row['p_sim_BB']:.3f})")

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR + 'morans_i_errors_all.csv', index=False)

tab = res[res['band_km'] == TABLE_BAND_KM].sort_values('n_errors')[
    ['Model', 'n_errors', 'error_rate', 'I', 'E_I', 'z_sim', 'p_sim', 'BB_obs', 'BB_exp', 'p_sim_BB']]
tab = tab.assign(error_rate=(100 * tab['error_rate']).round(2)).round(4)
tab.to_csv(OUT_DIR + 'morans_i_errors_table.csv', index=False)
print(f'\n=== Table ({TABLE_BAND_KM} km band) ===\n', tab.to_string(index=False))

# ------------------------------------------------------------------ correlogram
COLORS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for m, c in zip(MAIN_MODELS, COLORS):
    s = res[(res['Model'] == m) & res['I'].notna()].sort_values('band_km')
    ax.plot(s['band_km'], s['I'], color=c, lw=1.3, label=m)
    sig = s['p_sim'] < 0.05
    ax.scatter(s.loc[sig, 'band_km'], s.loc[sig, 'I'], color=c, s=30, zorder=3)
    ax.scatter(s.loc[~sig, 'band_km'], s.loc[~sig, 'I'], facecolor='white', edgecolor=c, s=30, zorder=3)
ax.axhline(-1 / (n - 1), color='grey', lw=0.8, ls='--', label='E[I]')
ax.set_xlabel('Distance band (km)', fontsize=9)
ax.set_ylabel("Moran's I of errors", fontsize=9)
ax.set_xticks(BANDS_KM)
ax.tick_params(labelsize=8)
ax.grid(alpha=0.3, lw=0.5)
ax.legend(fontsize=8, frameon=False)
ax.set_title('Filled: p < 0.05 (999 permutations)', fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR + 'Fig_morans_i_correlogram.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig_morans_i_correlogram.pdf', bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------ error map
err_map = errors[MAP_MODEL].astype(bool)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(coords[~err_map, 0] / 1000, coords[~err_map, 1] / 1000, s=5, color='#BBBBBB',
           label=f'Correct (n = {(~err_map).sum()})')
ax.scatter(coords[err_map, 0] / 1000, coords[err_map, 1] / 1000, s=18, color='#D55E00',
           edgecolor='black', lw=0.4, label=f'Misclassified (n = {err_map.sum()})')
ax.set_aspect('equal')
ax.set_xlabel('Easting (km)', fontsize=9)
ax.set_ylabel('Northing (km)', fontsize=9)
ax.tick_params(labelsize=8)
ax.legend(fontsize=8, frameon=False, loc='best')
ax.set_title(f'{MAP_MODEL} test-set errors ({gdf.crs.name})', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR + f'Fig_error_map_{MAP_MODEL}.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + f'Fig_error_map_{MAP_MODEL}.pdf', bbox_inches='tight')
plt.show()

# Which classes do the errors belong to? (helps interpret any clustering)
print('\nErrors by true class:')
by_class = pd.DataFrame({m: pd.Series(errors[m]).groupby(y_true).sum() for m in MAIN_MODELS})
by_class.index = [CLASS_NAMES.get(c, c) for c in by_class.index]
by_class['test points'] = pd.Series(y_true).value_counts().rename(index=CLASS_NAMES)
print(by_class)
by_class.to_csv(OUT_DIR + 'errors_by_class.csv')
print('\nFiles saved to:', OUT_DIR)